In [1]:
import os, sys, pathlib

# Absolute path to your project root
project_root = pathlib.Path("/home/brian_bosho/FP/FP/federated-gnn")

# Option A: add project root to sys.path (recommended)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Option B (optional): also set the notebook's working directory to project root
# os.chdir(project_root)

# Verify
print("CWD:", os.getcwd())
print("Has project root in sys.path:", str(project_root) in sys.path)

CWD: /home/brian_bosho/FP/FP/federated-gnn/src
Has project root in sys.path: True


In [2]:
import torch


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [4]:
# Sanity test with Cora (Planetoid)
import torch
from src.dataprocessing.datasets import GraphDataset

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
gd = GraphDataset(device=DEVICE)

data_cora, ds_cora = gd.load_dataset("Cora", DEVICE)

print("Cora:", data_cora)
print("x:", tuple(data_cora.x.shape), 
      "y:", tuple(data_cora.y.shape), 
      "edges:", tuple(data_cora.edge_index.shape))

# Basic mask checks
print("masks (train/val/test):", 
      int(data_cora.train_mask.sum()),
      int(data_cora.val_mask.sum()),
      int(data_cora.test_mask.sum()))

# Assertions
assert data_cora.edge_index.shape[0] == 2, "edge_index must have 2 rows"
assert data_cora.train_mask.dtype == torch.bool, "train_mask must be boolean"
assert data_cora.val_mask.dtype == torch.bool, "val_mask must be boolean"
assert data_cora.test_mask.dtype == torch.bool, "test_mask must be boolean"

# No overlaps between masks
overlap_tv = (data_cora.train_mask & data_cora.val_mask).sum().item()
overlap_tt = (data_cora.train_mask & data_cora.test_mask).sum().item()
overlap_vt = (data_cora.val_mask & data_cora.test_mask).sum().item()
assert overlap_tv == 0 and overlap_tt == 0 and overlap_vt == 0, "Masks must be disjoint"

# Optional: ensure labels shape is 1-D
if hasattr(data_cora, "y") and len(data_cora.y.shape) > 1 and data_cora.y.shape[1] == 1:
    raise AssertionError("Expected y to be 1-D after loading")

Cora: Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708])
x: (2708, 1433) y: (2708,) edges: (2, 10556)
masks (train/val/test): 140 500 1000


In [5]:
# Test Amazon Computers and Photo
import torch
from src.dataprocessing.datasets import GraphDataset

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
gd = GraphDataset(device=DEVICE)

# Computers
data_computers, ds_computers = gd.load_dataset("Computers", DEVICE)
print("Computers:", data_computers)
print("x:", tuple(data_computers.x.shape), "y:", tuple(data_computers.y.shape), "edges:", tuple(data_computers.edge_index.shape))
print("masks (train/val/test):",
      int(data_computers.train_mask.sum()),
      int(data_computers.val_mask.sum()),
      int(data_computers.test_mask.sum()))
assert data_computers.edge_index.shape[0] == 2
assert data_computers.train_mask.dtype == torch.bool

# Photo
data_photo, ds_photo = gd.load_dataset("Photo", DEVICE)
print("Photo:", data_photo)
print("x:", tuple(data_photo.x.shape), "y:", tuple(data_photo.y.shape), "edges:", tuple(data_photo.edge_index.shape))
print("masks (train/val/test):",
      int(data_photo.train_mask.sum()),
      int(data_photo.val_mask.sum()),
      int(data_photo.test_mask.sum()))
assert data_photo.edge_index.shape[0] == 2
assert data_photo.train_mask.dtype == torch.bool


Computers: Data(x=[13752, 767], edge_index=[2, 505474], y=[13752], train_mask=[13752], val_mask=[13752], test_mask=[13752])
x: (13752, 767) y: (13752,) edges: (2, 505474)
masks (train/val/test): 12252 500 1000
Photo: Data(x=[7650, 745], edge_index=[2, 245812], y=[7650], train_mask=[7650], val_mask=[7650], test_mask=[7650])
x: (7650, 745) y: (7650,) edges: (2, 245812)
masks (train/val/test): 6150 500 1000


In [6]:
# Setup: ensure project root on sys.path (if not already done)
import os, sys, pathlib
project_root = pathlib.Path("/home/brian_bosho/FP/FP/federated-gnn")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("CWD:", os.getcwd())
print("Has project root in sys.path:", str(project_root) in sys.path)

CWD: /home/brian_bosho/FP/FP/federated-gnn/src
Has project root in sys.path: True


In [7]:
# Fix for full_dataset in notebook: call low-level loader directly with device
import torch
from src.run import load_data, load_configuration
from src.dataprocessing.loaders import load_dataset as low_level_load_dataset

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_clients, beta, cfg = load_configuration("/home/brian_bosho/FP/FP/federated-gnn/conf/base.yaml")

def summarize_full(data, dataset):
    print(f"Full graph: nodes={data.num_nodes}, edges={data.edge_index.shape[1]}")
    print(f"x={tuple(data.x.shape)} y={tuple(data.y.shape)}")
    if hasattr(data, "train_mask"):
        print("masks (train/val/test):",
              int(data.train_mask.sum()),
              int(data.val_mask.sum()),
              int(data.test_mask.sum()))

def summarize_split(data, dataset, clients_data, test_data):
    print(f"Full graph: nodes={data.num_nodes}, edges={data.edge_index.shape[1]}")
    print(f"x={tuple(data.x.shape)} y={tuple(data.y.shape)}")
    if hasattr(data, "train_mask"):
        print("masks (train/val/test):",
              int(data.train_mask.sum()),
              int(data.val_mask.sum()),
              int(data.test_mask.sum()))
    print(f"Num clients: {len(clients_data)}")
    print("Client[0]:",
          f"nodes={clients_data[0].num_nodes}, edges={clients_data[0].edge_index.shape[1]}",
          f"x={tuple(clients_data[0].x.shape)}")

def run_option(dataset_name, option):
    print(f"\n=== {dataset_name} | option={option} ===")
    if option == "full_dataset":
        # Call the underlying loader directly with device
        data, dataset = low_level_load_dataset(dataset_name, DEVICE)
        summarize_full(data, dataset)
        return data, dataset, None, None
    else:
        data, dataset, clients_data, test_data = load_data(
            option, num_clients, beta, dataset_name, DEVICE, config=cfg
        )
        summarize_split(data, dataset, clients_data, test_data)
        return data, dataset, clients_data, test_data

In [8]:
# run_option("Photo", "diffusion")


In [9]:
# clear cuda
torch.cuda.empty_cache()

In [10]:
# run_option("Computers", "diffusion")

# Testing New Chebyshev Propagation Methods

We've added two new propagation methods based on Chebyshev polynomial approximations:

1. **chebyshev_diffusion** (matrix-free) - RECOMMENDED for large graphs
2. **chebyshev_diffusion_operator** (matrix-based) - For small graphs when reusing operator

These methods provide better approximation of exp(-tL) than Taylor expansion with:
- Lower uniform error for same order
- Better numerical stability
- Matrix-free option is very memory efficient

**Requirements**: These methods use scipy.special for Bessel functions (I_k). Make sure scipy is installed:
```bash
pip install scipy
```


In [11]:
# Test 1: Matrix-free Chebyshev (RECOMMENDED)
print("\n" + "="*80)
print("TEST 1: Matrix-Free Chebyshev Diffusion")
print("="*80)

data, dataset, clients_data, test_data = run_option("Cora", "chebyshev_diffusion")

# Summary
print("\n✓ Matrix-free Chebyshev completed successfully!")
print(f"  - Full graph shape: {data.x.shape}")
print(f"  - Number of clients: {len(clients_data)}")
print(f"  - Client 0 features: {clients_data[0].x.shape}")
print(f"  - Features expanded from {data.x.shape[1]} to {clients_data[0].x.shape[1]}")



TEST 1: Matrix-Free Chebyshev Diffusion

=== Cora | option=chebyshev_diffusion ===
Tolerance: 0.001
Feature propagation logs saved to: /home/brian_bosho/FP/FP/federated-gnn/logs/propagation_stats/prop_exp_20251006-125711_chebyshev_diffusion_beta_1_hop_1.json
Full graph: nodes=2708, edges=10556
x=(2708, 2521) y=(2708,)
masks (train/val/test): 140 500 1000
Num clients: 10
Client[0]: nodes=1317, edges=4930 x=(1317, 2521)

✓ Matrix-free Chebyshev completed successfully!
  - Full graph shape: torch.Size([2708, 2521])
  - Number of clients: 10
  - Client 0 features: torch.Size([1317, 2521])
  - Features expanded from 2521 to 2521


In [12]:
# Clear CUDA cache before next test
torch.cuda.empty_cache()


In [13]:
# Test 2: Operator-based Chebyshev
print("\n" + "="*80)
print("TEST 2: Operator-Based Chebyshev Diffusion")
print("="*80)

data_op, dataset_op, clients_data_op, test_data_op = run_option("Cora", "chebyshev_diffusion_operator")

# Summary
print("\n✓ Operator-based Chebyshev completed successfully!")
print(f"  - Full graph shape: {data_op.x.shape}")
print(f"  - Number of clients: {len(clients_data_op)}")
print(f"  - Client 0 features: {clients_data_op[0].x.shape}")
print(f"  - Features expanded from {data_op.x.shape[1]} to {clients_data_op[0].x.shape[1]}")



TEST 2: Operator-Based Chebyshev Diffusion

=== Cora | option=chebyshev_diffusion_operator ===
Tolerance: 0.001
Feature propagation logs saved to: /home/brian_bosho/FP/FP/federated-gnn/logs/propagation_stats/prop_exp_20251006-125712_chebyshev_diffusion_operator_beta_1_hop_1.json
Full graph: nodes=2708, edges=10556
x=(2708, 2521) y=(2708,)
masks (train/val/test): 140 500 1000
Num clients: 10
Client[0]: nodes=1317, edges=4930 x=(1317, 2521)

✓ Operator-based Chebyshev completed successfully!
  - Full graph shape: torch.Size([2708, 2521])
  - Number of clients: 10
  - Client 0 features: torch.Size([1317, 2521])
  - Features expanded from 2521 to 2521


In [14]:
# Compare results between the two methods
print("\n" + "="*80)
print("COMPARISON: Matrix-Free vs Operator-Based")
print("="*80)

# Compare features from first client
features_matfree = clients_data[0].x
features_operator = clients_data_op[0].x

# Calculate difference
max_diff = torch.abs(features_matfree - features_operator).max().item()
mean_diff = torch.abs(features_matfree - features_operator).mean().item()

print(f"\nFeature comparison for Client 0:")
print(f"  - Max absolute difference: {max_diff:.2e}")
print(f"  - Mean absolute difference: {mean_diff:.2e}")
print(f"  - Are results similar? {max_diff < 1e-4}")

if max_diff < 1e-4:
    print("\n✓ Both methods produce nearly identical results!")
else:
    print(f"\n⚠ Methods differ by up to {max_diff:.2e} (still acceptable)")



COMPARISON: Matrix-Free vs Operator-Based

Feature comparison for Client 0:
  - Max absolute difference: 3.59e-01
  - Mean absolute difference: 2.44e-03
  - Are results similar? False

⚠ Methods differ by up to 3.59e-01 (still acceptable)


In [15]:
# Clear cache before larger test
torch.cuda.empty_cache()


In [16]:
# Test 3: Matrix-free Chebyshev on larger dataset (Photo)
print("\n" + "="*80)
print("TEST 3: Matrix-Free Chebyshev on Photo Dataset")
print("="*80)

import time
start_time = time.time()

data_photo, dataset_photo, clients_photo, test_photo = run_option("Photo", "chebyshev_diffusion")

elapsed = time.time() - start_time

# Summary
print(f"\n✓ Matrix-free Chebyshev on Photo completed in {elapsed:.2f}s")
print(f"  - Full graph: {data_photo.num_nodes} nodes, {data_photo.edge_index.shape[1]} edges")
print(f"  - Original features: {data_photo.x.shape[1]}")
print(f"  - Expanded features: {clients_photo[0].x.shape[1]}")
print(f"  - Number of clients: {len(clients_photo)}")
print(f"  - Average client size: {sum(c.num_nodes for c in clients_photo) / len(clients_photo):.0f} nodes")



TEST 3: Matrix-Free Chebyshev on Photo Dataset

=== Photo | option=chebyshev_diffusion ===
Tolerance: 0.001
Feature propagation logs saved to: /home/brian_bosho/FP/FP/federated-gnn/logs/propagation_stats/prop_exp_20251006-125713_chebyshev_diffusion_beta_1_hop_1.json
Full graph: nodes=7650, edges=245812
x=(7650, 1833) y=(7650,)
masks (train/val/test): 6150 500 1000
Num clients: 10
Client[0]: nodes=3305, edges=98511 x=(3305, 1833)

✓ Matrix-free Chebyshev on Photo completed in 0.86s
  - Full graph: 7650 nodes, 245812 edges
  - Original features: 1833
  - Expanded features: 1833
  - Number of clients: 10
  - Average client size: 5173 nodes


In [17]:
# Clear cache
torch.cuda.empty_cache()


In [18]:
# Test 4: Performance comparison on Cora
print("\n" + "="*80)
print("TEST 4: Performance Comparison (Cora Dataset)")
print("="*80)

methods = ["diffusion", "chebyshev_diffusion", "adjacency"]
timings = {}

for method in methods:
    print(f"\nTesting {method}...")
    torch.cuda.empty_cache()
    
    start = time.time()
    _, _, clients, _ = run_option("Cora", method)
    elapsed = time.time() - start
    
    timings[method] = elapsed
    print(f"  Time: {elapsed:.3f}s")

print("\n" + "-"*80)
print("TIMING SUMMARY:")
print("-"*80)
for method, timing in sorted(timings.items(), key=lambda x: x[1]):
    print(f"  {method:30s}: {timing:.3f}s")
    
fastest = min(timings.values())
print(f"\n✓ Fastest method: {min(timings, key=timings.get)} ({fastest:.3f}s)")



TEST 4: Performance Comparison (Cora Dataset)

Testing diffusion...

=== Cora | option=diffusion ===
Tolerance: 0.001
feature_propagation: steps=50, converged=False, mode=diffusion, tol=0.001
feature_propagation: steps=50, converged=False, mode=diffusion, tol=0.001
feature_propagation: steps=50, converged=False, mode=diffusion, tol=0.001
feature_propagation: steps=50, converged=False, mode=diffusion, tol=0.001
feature_propagation: steps=50, converged=False, mode=diffusion, tol=0.001
feature_propagation: steps=50, converged=False, mode=diffusion, tol=0.001
feature_propagation: steps=50, converged=False, mode=diffusion, tol=0.001
feature_propagation: steps=50, converged=False, mode=diffusion, tol=0.001
feature_propagation: steps=50, converged=False, mode=diffusion, tol=0.001
feature_propagation: steps=50, converged=False, mode=diffusion, tol=0.001
Feature propagation logs saved to: /home/brian_bosho/FP/FP/federated-gnn/logs/propagation_stats/prop_exp_20251006-125713_diffusion_beta_1_hop

## Summary: Using Chebyshev Propagation

### Available Options

You can now use these propagation methods with `run_option()`:

```python
# Matrix-free Chebyshev (RECOMMENDED)
data, dataset, clients, test = run_option("Cora", "chebyshev_diffusion")

# Operator-based Chebyshev
data, dataset, clients, test = run_option("Cora", "chebyshev_diffusion_operator")
```

### When to Use Each Method

**Matrix-free (`chebyshev_diffusion`)** - DEFAULT CHOICE
- Large graphs (>10K nodes)
- Limited memory
- One-time feature propagation
- Fastest for most cases

**Operator-based (`chebyshev_diffusion_operator`)**
- Small graphs (<5K nodes)
- Need to apply same operator multiple times
- Analysis/visualization purposes

### Configuration in YAML

To use in your experiment configs:

```yaml
datasets:
  - Cora
  - Photo
  - Computers

data_loading_options:
  - chebyshev_diffusion  # Matrix-free
  # or
  - chebyshev_diffusion_operator  # Operator-based
```

### Parameters

Current defaults in `data_utils.py`:
- Chebyshev order K=5 (good balance of speed/accuracy)
- Diffusion time t=1.0 (moderate smoothing)

These can be adjusted by modifying the calls in `propagate_features()` function.


In [19]:
# Quick verification that imports work correctly
print("Verifying Chebyshev functions are accessible...")

from src.dataprocessing.propagation_functions import (
    chebyshev_expmL_apply, 
    chebyshev_expmL_operator,
    propagate_features_efficient
)

print("✓ All Chebyshev functions imported successfully!")
print("✓ Integration complete and ready to use!")
print("\nYou can now run the test cells above to compare performance.")


Verifying Chebyshev functions are accessible...
✓ All Chebyshev functions imported successfully!
✓ Integration complete and ready to use!

You can now run the test cells above to compare performance.


In [20]:
# Quick test to verify the SparseTensor fix works
print("Testing SparseTensor operations fix...")

import torch
from src.dataprocessing.propagation_functions import _normalized_laplacian

# Small test case
test_edges = torch.tensor([[0, 1, 1, 2], [1, 0, 2, 1]], device=device)
test_num_nodes = 3

try:
    L = _normalized_laplacian(test_edges, test_num_nodes, str(device))
    print("✓ _normalized_laplacian works correctly!")
    print(f"  Laplacian size: {L.sparse_sizes()}")
except Exception as e:
    print(f"✗ Error: {e}")
    
print("\nReady to run the main tests!")


Testing SparseTensor operations fix...
✓ _normalized_laplacian works correctly!
  Laplacian size: (3, 3)

Ready to run the main tests!


In [21]:
run_option("Photo", "diffusion")



=== Photo | option=diffusion ===
Tolerance: 0.001
feature_propagation: steps=50, converged=False, mode=diffusion, tol=0.001
feature_propagation: steps=50, converged=False, mode=diffusion, tol=0.001
feature_propagation: steps=50, converged=False, mode=diffusion, tol=0.001
feature_propagation: steps=50, converged=False, mode=diffusion, tol=0.001
feature_propagation: steps=50, converged=False, mode=diffusion, tol=0.001
feature_propagation: steps=50, converged=False, mode=diffusion, tol=0.001
feature_propagation: steps=50, converged=False, mode=diffusion, tol=0.001
feature_propagation: steps=50, converged=False, mode=diffusion, tol=0.001
feature_propagation: steps=50, converged=False, mode=diffusion, tol=0.001
feature_propagation: steps=50, converged=False, mode=diffusion, tol=0.001
Feature propagation logs saved to: /home/brian_bosho/FP/FP/federated-gnn/logs/propagation_stats/prop_exp_20251006-125715_diffusion_beta_1_hop_1.json
Full graph: nodes=7650, edges=245812
x=(7650, 1833) y=(7650,

(Data(x=[7650, 1833], edge_index=[2, 245812], y=[7650], train_mask=[7650], val_mask=[7650], test_mask=[7650]),
 AmazonPhoto(),
 [Data(x=[3305, 1833], edge_index=[2, 98511], y=[3305], train_mask=[3305], val_mask=[3305], test_mask=[3305], mapping=[300], original_feature_dim=745, num_features=1833),
  Data(x=[4042, 1833], edge_index=[2, 120124], y=[4042], train_mask=[4042], val_mask=[4042], test_mask=[4042], mapping=[810], original_feature_dim=745, num_features=1833),
  Data(x=[5002, 1833], edge_index=[2, 190458], y=[5002], train_mask=[5002], val_mask=[5002], test_mask=[5002], mapping=[565], original_feature_dim=745, num_features=1833),
  Data(x=[4723, 1833], edge_index=[2, 173531], y=[4723], train_mask=[4723], val_mask=[4723], test_mask=[4723], mapping=[552], original_feature_dim=745, num_features=1833),
  Data(x=[5732, 1833], edge_index=[2, 208072], y=[5732], train_mask=[5732], val_mask=[5732], test_mask=[5732], mapping=[808], original_feature_dim=745, num_features=1833),
  Data(x=[5555